# Path C+ Option C — Full retrain Stage 1 + Stage 2 from-scratch (3 seeds)

**Goal** : produce a thesis-grade Path C+ Oracle retrain matching CorrDiff Normal V2 config (`training_config_corrdiff_normal.yaml`, Stage 1 = 15 epochs, Stage 2 = 200 epochs), with Path C+ hyperparameter overrides, K9 temporal split, 3 seeds, and full H1-H5 evaluation pipeline mirrored from `st_cdgm_noncausal_training.ipynb` (see `path_c_plus/audit/NONCAUSAL_EVAL_CONTRACT.md`).

**Code path** : `src.st_cdgm.training.training_loop.train_epoch_stage1` + `train_epoch_stage2` (NOT `finetune_bundle_b` like A1).

**Hyperparameter strategy** : structural CorrDiff config (epochs / batch / scheduler) + Path C+ overrides (lambda_dag_prior=0.40, lambda_l1=0.04->0.005, g_phys_alpha=0.25, dag_grad_gate auto-scale ramp). See `PATHCPLUS_HYPERPARAM_OVERRIDES` in `path_c_plus.scripts.option_c_helpers`.

**Output structure** :
```
/content/drive/MyDrive/climate_data/oracle_full/
  seed_42/
    epoch_last.pth                        (resume)
    final_validation_metrics.json         (cell 61 mirror)
    domain_metrics.json                   (cell 62 mirror)
    eval_samples.npz                      (cell 63 mirror)
    aligned_metrics_ACCESS-CM2_causal.json (cell 64+65, in-dist)
    aligned_metrics_EC-Earth3_causal.json   (cell 64+65, OOD)
    aligned_metrics_NorESM2-MM_causal.json  (cell 64+65, OOD)
    results.json                          (Path C+ specific: Q_phys + verdict)
  seed_7/  ...
  seed_123/ ...
  oracle_full_aggregate.json              (cross-seed H1-H5 verdict + Holm-Bonferroni)
```

**Baseline for H2-H5** : `/content/drive/MyDrive/climate_data/ckpt_noncausal/` (epoch=200, verified COMPLETE)

**Resume semantics** : each seed's training writes `epoch_last.pth` per epoch atomically. If `oracle_full/seed_<n>/results.json` exists, the seed is skipped. To re-run a seed, delete the directory.

**Estimated compute** : 30-50h/seed on A100 (Stage 1 ~3-5h, Stage 2 ~25-45h), 3 seeds = 90-150h sequential. Use Colab Pro+ A100, 1-2 sessions per seed.

In [ ]:
# >>> Cell 1 : Bootstrap Colab + git sync
import os, sys, subprocess, time, shlex
from pathlib import Path

GIT_URL: str | None = "https://github.com/leonelkenfack/stcdgm.git"
GIT_BRANCH: str = "four-node-causal"
LOCAL_PROJECT = "/content/climate_data"
GIT_PULL_ON_RESUME = True
SKIP_PIP_IF_IMPORTABLE = True

_IS_COLAB = "google.colab" in sys.modules or Path("/content").exists()

def _run(cmd: str, *, check: bool = True, timeout: int | None = None) -> int:
    print(f"$ {cmd}"); t0 = time.time()
    rc = subprocess.call(shlex.split(cmd), timeout=timeout)
    dt = time.time() - t0; print(f"  rc={rc}  ({dt:.1f}s)")
    if check and rc != 0:
        raise RuntimeError(f"Commande échouée : {cmd!r} (rc={rc})")
    return rc

if _IS_COLAB:
    _T0 = time.time()
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
    project_path = Path(LOCAL_PROJECT)
    if not (project_path / ".git").exists():
        project_path.parent.mkdir(parents=True, exist_ok=True)
        _run(f"git clone --depth 1 -b {GIT_BRANCH} {GIT_URL} {LOCAL_PROJECT}")
    if GIT_PULL_ON_RESUME:
        _run(f"git -C {LOCAL_PROJECT} fetch --depth=200 origin {GIT_BRANCH}", timeout=180)
        try:
            cur = subprocess.check_output(shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --abbrev-ref HEAD")).decode().strip()
        except Exception:
            cur = ""
        if cur != GIT_BRANCH:
            _run(f"git -C {LOCAL_PROJECT} checkout -B {GIT_BRANCH} origin/{GIT_BRANCH}", timeout=30)
        else:
            _run(f"git -C {LOCAL_PROJECT} reset --hard origin/{GIT_BRANCH}", timeout=30)
        head_sha = subprocess.check_output(shlex.split(f"git -C {LOCAL_PROJECT} rev-parse --short HEAD")).decode().strip()
        head_msg = subprocess.check_output(shlex.split(f"git -C {LOCAL_PROJECT} log -1 --pretty=%s")).decode().strip()
        print(f"   HEAD = {head_sha}  ({head_msg})")
    os.chdir(project_path)
    src_path = str(project_path / "src")
    if src_path not in sys.path:
        sys.path.insert(0, src_path)
    if SKIP_PIP_IF_IMPORTABLE:
        try:
            import st_cdgm  # noqa: F401
            from omegaconf import OmegaConf  # noqa: F401
            from diffusers import UNet2DConditionModel  # noqa: F401
            import torch_geometric  # noqa: F401
            print("OK Imports critiques — pip install sauté.")
        except ImportError as e:
            print(f"pip install requis : {e}")
            EXTRA_DEPS = ["omegaconf==2.3.0", "hydra-core==1.3.2", "diffusers==0.36.0",
                          "transformers==4.57.6", "accelerate==1.12.0", "huggingface-hub==0.36.0",
                          "safetensors==0.7.0", "xbatcher", "webdataset", "cftime", "h5netcdf",
                          "numcodecs", "torch-geometric", "xformers"]
            _run(f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
                 + " ".join(shlex.quote(p) for p in EXTRA_DEPS), timeout=600)
            _run(f"{shlex.quote(sys.executable)} -m pip install --no-warn-script-location "
                 f"--no-deps -e {LOCAL_PROJECT}", timeout=120)
    print(f"\nBootstrap terminé en {time.time() - _T0:.1f}s.")
else:
    _here = Path.cwd()
    for _c in [_here, *_here.parents]:
        if (_c / "config" / "training_config.yaml").exists() and (_c / "setup.py").exists():
            if _c != _here:
                os.chdir(_c)
            break
    print("Hors Colab")


In [ ]:
# >>> Cell 2 : Config CorrDiff Normal V2 + Path C+ hyperparam override
from omegaconf import OmegaConf
from path_c_plus.scripts.option_c_helpers import (
    PATHCPLUS_HYPERPARAM_OVERRIDES,
    PC13_NEW_PARAMS_ALLOWLIST,
)
from path_c_plus.scripts.gpu_detect import detect_gpu_profile, print_profile_banner

GPU_PROFILE = detect_gpu_profile()
print_profile_banner(GPU_PROFILE)
if GPU_PROFILE.get("profile_id") not in ("a100", "h100"):
    print(f"\nWARN Option C was budgeted for A100. Detected: {GPU_PROFILE.get('profile_id')}")
    print("     T4/V100 will multiply wall-time by 5-8x.")

# Load CorrDiff Normal V2 config (causal variant)
CONFIG = OmegaConf.load("config/training_config.yaml")
_corrdiff = OmegaConf.load("config/training_config_corrdiff_normal.yaml")
CONFIG = OmegaConf.merge(CONFIG, _corrdiff)

# Apply GPU profile (batch_size, amp, num_workers)
CONFIG.training.batch_size = GPU_PROFILE["batch_size"]
CONFIG.training.use_amp = GPU_PROFILE["use_amp"]
CONFIG.training.num_workers = GPU_PROFILE["num_workers"]

# === Path C+ hyperparameter override ===
# These replace V5-mini defaults (lambda_dag_prior=0.05 etc.) with Path C+
# values that A1 used and produced Q_phys_cont = 0.521. The structural CorrDiff
# settings (15+200 epochs, stride=2, batch=64, scheduler edm_karras) are
# preserved. Only the DAG-loss hyperparameters are overridden.
print("\n=== Path C+ hyperparameter override (vs CorrDiff Normal V2 defaults) ===")
ts_cfg = CONFIG.two_stage
for key, new_val in PATHCPLUS_HYPERPARAM_OVERRIDES.items():
    if key in ts_cfg.get("stage1", {}):
        old_val = ts_cfg.stage1.get(key)
        ts_cfg.stage1[key] = new_val
        print(f"  stage1.{key:30s} : {old_val} -> {new_val}")
    else:
        # Inject as a new key (e.g., lambda_l1_start / lambda_l1_end are A1 names)
        ts_cfg.stage1[key] = new_val
        print(f"  stage1.{key:30s} : <NEW> = {new_val}")

# Confirm run_variant = causal
assert ts_cfg.get("run_variant") == "causal", (
    f"run_variant must be 'causal' for Option C, got {ts_cfg.get('run_variant')!r}"
)
print(f"\n  run_variant = {ts_cfg.run_variant}")
print(f"  stage1.epochs_max = {ts_cfg.stage1.get('epochs_max')}")
print(f"  stage2.epochs_max = {ts_cfg.stage2.get('epochs_max')}")
print(f"  data.stride = {CONFIG.data.stride}")
print(f"  training.batch_size = {CONFIG.training.batch_size}")

# ORACLE_FULL_DIR (Path C+ Option C output)
ORACLE_FULL_DIR = Path("/content/drive/MyDrive/climate_data/oracle_full")
ORACLE_FULL_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n[Option C] ORACLE_FULL_DIR = {ORACLE_FULL_DIR}")
print(f"[Option C] Existing seed dirs : {sorted(p.name for p in ORACLE_FULL_DIR.iterdir() if p.is_dir())}")

# Noncausal baseline (for H2-H5)
CKPT_NONCAUSAL_DIR = Path("/content/drive/MyDrive/climate_data/ckpt_noncausal")
assert CKPT_NONCAUSAL_DIR.exists(), f"Noncausal baseline missing : {CKPT_NONCAUSAL_DIR}"
print(f"[Option C] CKPT_NONCAUSAL_DIR = {CKPT_NONCAUSAL_DIR}  (H2-H5 baseline)")
